# Assignment 7: Classification Analysis

**Goal:** build and compare a **Decision Tree** and a **Naive Bayes** classifier on your
group's dataset, then report what you found.

The structure and hints are provided; Write the code
in the cells marked `# TODO`. If you get stuck on syntax, the `classification_code_example.ipynb`
from this week shows every step worked out on an example dataset.

> A sample dataset (`data/loan_sample.csv`) is included so the template runs immediately.
> **Replace it with your chosen Kaggle dataset** and update the target column.

## Your dataset

| Group dataset | Example classification task | Target column | Kaggle link |
|---|---|---|---|
| SF Crime | Predict crime category, or district | `Category` / `PdDistrict` | https://www.kaggle.com/competitions/sf-crime/data
| NASA Asteroids | Predict if an asteroid is hazardous | `Hazardous` | https://www.kaggle.com/datasets/shrutimehta/nasa-asteroids-classification
| Hotel Booking | Predict if a booking is cancelled, or hotel type | `is_canceled` / `hotel` | https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand

Use a Decision Tree for one task and Naive Bayes for the other (or, for Asteroids, run
both on the same task and compare). Keep notes as you go; you will need them for the report
at the bottom.

In [ ]:
# Prerequisites
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

print("Ready.")

## Step 1 - Data exploration

Load your data and get to know it: how many rows and columns, what the columns mean, and
how balanced the classes are. Class balance matters: if 90% of rows are one class, a model
can look accurate just by guessing that class every time.

In [ ]:
# GIVEN: load the dataset. Change this path to your group's file when you have it.
df = pd.read_csv("nasa.csv")

# Set this to the name of the column you are predicting.
target = "Hazardous"

df.head()

In [ ]:
# TODO: explore the data

# how big is the dataset
print("Dataset Shape:")
df.shape # --> (4687, 40)

# what columns exist
print("\nDataset Info:")
df.info()

# how many hazardous asteroids are there
print("\nHazardous Counts:")
df[target].value_counts() # --> F: 3932, T: 755

# gives out statisticsor numeric values
print("\nNumerical Summary")
df.describe()

## Step 2 - Preprocessing

Get the data ready for the models. You will typically need to:

- Drop columns that are just identifiers or are not useful (e.g. an ID column).
- Handle missing values (fill numeric columns with the median, text columns with the mode).
- Encode categorical (text) columns into numbers with `pd.get_dummies(...)` or `LabelEncoder`.
- Separate the target `y` from the features `X`, then split into train and test sets.

In [ ]:
# TODO: drop unhelpful columns and handle missing values

# dropping "ID" and "Name" columns from the astriods
data = df.drop(columns=["Neo Reference ID", "Name"]).copy()

# fills in any missing data or values with the median, otherwise the mode
for col in data.columns:
    if pd.api.types.is_numeric_dtype(data[col]):
        data[col] = data[col].fillna(data[col].median())
    else:
        data[col] = data[col].fillna(data[col].mode()[0])

In [ ]:
# TODO: build X (features) and y (target), then encode the categorical columns

y = data["Hazardous"]

# 'pd.get_dummies()' conver text into machine-learning-friendy numbers
X = pd.get_dummies(data.drop(columns=[target]),drop_first=True)

In [ ]:
# TODO: split into training and test sets

# text_size = 25% testing (75% training)
# random_state=1 = shuffle the asteroids the same way every time
# stratify=y = both groups keep similar precentages
X_train, X_test, y_train, y_test = train_test_split(
          X, y, test_size=0.25, random_state=1, stratify=y)

## Step 3 - Build the models

Each classifier is only a few lines: create it, then call `.fit(X_train, y_train)`.

In [ ]:
# TODO: train a Decision Tree

# creates an empty Decision Tree model
dtree = DecisionTreeClassifier(max_depth=5, random_state=1)

# studies the train astroids and learn patterns
dtree.fit(X_train, y_train)

In [ ]:
# TODO: train a Naive Bayes classifier

# creates a Naive Bayes model
nb_model = GaussianNB()

# studies the train astroids and learn patterns
nb_model.fit(X_train, y_train)

## Step 4 - Evaluate and compare

For **each** model, predict on the test set and report:

- accuracy (`accuracy_score`)
- the confusion matrix (`ConfusionMatrixDisplay.from_predictions`)
- precision, recall, and F1 (`classification_report`)

Then compare: which model is more accurate, and what do the precision and recall for the
class you care about tell you?

In [ ]:
# TODO: evaluate the Decision Tree
dt_pred = dtree.predict(X_test)

print("DECISION TREE RESULTS")
print("Accuracy:", accuracy_score(y_test, dt_pred))

print("\nConfusion Matrix:")
ConfusionMatrixDisplay.from_predictions(
    y_test, dt_pred,
    labels=["F", "T"],
    display_labels=["Not Hazardous", "Hazardous"],
    cmap="Blues"
)
plt.show()

print("\nClassification Report:")
print(classification_report(
    y_test, dt_pred,
    labels=["F", "T"],
    target_names=["Not Hazardous", "Hazardous"],
    zero_division=0
))

In [ ]:
# TODO: evaluate the Naive Bayes model (same steps as above)
nb_pred = nb_model.predict(X_test)

print("\nNAIVE BAYES RESULTS")
print("Accuracy:", accuracy_score(y_test, nb_pred))

print("\nConfusion Matrix:")
ConfusionMatrixDisplay.from_predictions(
    y_test, nb_pred,
    labels=["F", "T"],
    display_labels=["Not Hazardous", "Hazardous"],
    cmap="Blues"
)
plt.show()

print("\nClassification Report:")
print(classification_report(
    y_test, nb_pred,
    labels=["F", "T"],
    target_names=["Not Hazardous", "Hazardous"],
    zero_division=0
))



In [ ]:
# TODO: compare the two models' accuracy against a baseline
baseline = y_test.value_counts(normalize=True).max()
dt_accuracy = accuracy_score(y_test, dt_pred)
nb_accuracy = accuracy_score(y_test, nb_pred)

print("\nBASELINE AND MODEL COMPARISON")
print(f"Baseline accuracy: {baseline:.4f} ({baseline * 100:.2f}%)")
print(f"Decision Tree accuracy: {dt_accuracy:.4f} ({dt_accuracy * 100:.2f}%)")
print(f"Naive Bayes accuracy: {nb_accuracy:.4f} ({nb_accuracy * 100:.2f}%)")

print("\nBest accuracy:", 
      "Decision Tree" if dt_accuracy > nb_accuracy
      else "Naive Bayes" if nb_accuracy > dt_accuracy
      else "Tie")

print("Decision Tree beats baseline:", dt_accuracy > baseline)
print("Naive Bayes beats baseline:", nb_accuracy > baseline)

Step 5:

## Step 5 - Findings

Write your answers here as you work.

1. **Preprocessing / data reduction** - what did you have to clean, drop, encode, or sample, and why?

   We removed the asteroid ID and name columns because they identify individual asteroids rather than provide useful measurements for predicting whether they are hazardous. We filled missing numerical values with their column medians and missing categorical values with their most common values. We also separated the Hazardous column as the target and used one-hot encoding to convert categorical predictors into numerical values. Finally, We split the data into 75% training data and 25% testing data while keeping the class proportions similar in both sets.

2. **Data used** - which features did you select as predictors, and why those?

   We used the available asteroid measurements, including physical characteristics and orbital measurements, as predictors. These features can help the models identify patterns associated with hazardous asteroids. We excluded the asteroid ID and name because they are identifiers rather than meaningful measurements. The Hazardous column was used as the target variable, with T representing hazardous and F representing not hazardous.

3. **Findings** - what did the analysis reveal about the data itself (patterns, surprises)?

   The dataset contained 4,687 asteroids and 40 columns before preprocessing. There were 3,932 non-hazardous asteroids and 755 hazardous asteroids, meaning that approximately 83.9% were classified as non-hazardous and 16.1% as hazardous. This class imbalance is important because a model could achieve about 83.9% accuracy by always predicting that an asteroid is not hazardous. Therefore, I need to consider precision and recall for the hazardous class, along with overall accuracy, to determine whether the models are useful.

4. **Decision Tree vs Naive Bayes** - which performed better on your task, and what are the
   strengths and shortcomings of each one here?

  The Decision Tree achieved an accuracy of [insert accuracy]%, while Naive Bayes achieved an accuracy of [insert accuracy]%. The model that performed better in terms of accuracy was [insert model]. However, I also compared precision and recall for the hazardous class because missing a hazardous asteroid is an important error. The hazardous-class recall was [insert Decision Tree recall] for the Decision Tree and [insert Naive Bayes recall] for Naive Bayes. The Decision Tree can learn nonlinear patterns and interactions between features, while Naive Bayes is relatively simple and fast but assumes that features are conditionally independent given the class. Overall, I compared both models against the majority-class baseline to determine whether they provided useful predictions beyond simply guessing the most common class.

5. **Challenges** - What was difficult, and how did you handle it?

   *One challenge was preparing the dataset for machine learning by handling missing values, removing unnecessary columns, and converting categorical data into numerical values. Another challenge was the imbalance between hazardous and non-hazardous asteroids, which can make accuracy misleading. I addressed these issues by filling missing values, using one-hot encoding, and splitting the data with stratification. I also evaluated both models using confusion matrices, precision, recall, and F1-scores so I could compare their ability to identify hazardous asteroids rather than relying on accuracy alone.

## Submission checklist

- This notebook (renamed `Last_First_Assignment7.ipynb`), with all cells run and output showing
- Link to your GitHub repository